In [1]:
from Bio import SeqIO
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import chi2

In [2]:
alphabet = ['A', 'C', 'G', 'T']
idx = {b: i for i, b in enumerate(alphabet)}

# =========================
# 1. Чтение реальной последовательности
# =========================
# Вариант для multi-FASTA без искусственных переходов между записями
records = list(SeqIO.parse("gene.fasta", "fasta"))

real_sequences = []
for record in records:
    seq = str(record.seq).upper()
    seq = ''.join(base for base in seq if base in "ACGT")
    if len(seq) >= 2:
        real_sequences.append(seq)

real_seq_concat = ''.join(real_sequences)


In [3]:
# 2. Генерация случайной последовательности 0-го порядка

mono_counts = Counter(real_seq_concat)
total_bases = sum(mono_counts[b] for b in alphabet)
mono_probs = np.array([mono_counts[b] / total_bases for b in alphabet])

rng = np.random.default_rng(42)
random_seq = ''.join(rng.choice(alphabet, size=len(real_seq_concat), p=mono_probs))



In [4]:
# 3. Функции для O, E, chi-square, p-value
def observed_dinuc_matrix(seq):
    O = np.zeros((4, 4), dtype=int)
    for i in range(len(seq) - 1):
        a, b = seq[i], seq[i+1]
        if a in idx and b in idx:
            O[idx[a], idx[b]] += 1
    return O

def expected_dinuc_matrix(seq):
    counts = Counter(seq)
    N_pairs = len(seq) - 1
    probs = np.array([counts[b] / len(seq) for b in alphabet])

    E = np.zeros((4, 4), dtype=float)
    for i in range(4):
        for j in range(4):
            E[i, j] = N_pairs * probs[i] * probs[j]
    return E

def chi_square_test_markov0(seq):
    O = observed_dinuc_matrix(seq)
    E = expected_dinuc_matrix(seq)

    chi2_stat = np.sum((O - E)**2 / E)
    p_value = chi2.sf(chi2_stat, df=9)

    return O, E, chi2_stat, p_value

In [5]:
# 4. Тест для реальной последовательности
# =========================
O_real, E_real, chi2_real, p_real = chi_square_test_markov0(real_seq_concat)



In [6]:

# 5. Тест для случайной последовательности 0-го порядка
# =========================
O_rand, E_rand, chi2_rand, p_rand = chi_square_test_markov0(random_seq)


In [7]:

# 6.вывод
def print_results(name, O, E, chi2_stat, p_value):
    print(f"\n=== {name} ===")

    print("\nObserved dinucleotide counts O:")
    print(pd.DataFrame(O, index=alphabet, columns=alphabet))

    print("\nExpected counts E:")
    print(pd.DataFrame(E, index=alphabet, columns=alphabet).round(2))

    print(f"\nChi-square statistic: {chi2_stat:.4f}")
    print(f"p-value: {p_value:.6e}")

    if p_value < 0.05:
        print("Conclusion: reject independence -> 1st-order model is needed.")
    else:
        print("Conclusion: independence not rejected -> 0th-order model may be sufficient.")

print_results("Real sequence", O_real, E_real, chi2_real, p_real)
print_results("Random 0th-order sequence", O_rand, E_rand, chi2_rand, p_rand)



=== Real sequence ===

Observed dinucleotide counts O:
       A      C      G       T
A  81545  38022  55155   70692
C  53043  36051   6905   58281
G  45861  31508  37444   48932
T  64965  48699  64241  106069

Expected counts E:
          A         C         G         T
A  71073.23  44680.15  47421.25  82240.08
C  44680.15  28088.15  29811.34  51700.18
G  47421.25  29811.34  31640.25  54871.96
T  82240.08  51700.18  54871.96  95161.44

Chi-square statistic: 36187.3818
p-value: 0.000000e+00
Conclusion: reject independence -> 1st-order model is needed.

=== Random 0th-order sequence ===

Observed dinucleotide counts O:
       A      C      G      T
A  71151  44692  47580  81927
C  44681  28231  29894  51382
G  47320  29878  31777  55172
T  82198  51388  54896  95246

Expected counts E:
          A         C         G         T
A  71035.59  44641.97  47525.08  82147.08
C  44641.97  28055.02  29866.90  51624.93
G  47525.08  29866.90  31795.80  54959.02
T  82147.08  51624.93  54959.02  94